In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import yaml
from datetime import timedelta
import matplotlib.pyplot as plt
from src.adapters.local_data import load_ohlc_from_csv
from src.core.trend_id import identify_trend

# Load original data
all_candles_15m = load_ohlc_from_csv("data/processed/R_10_15m.csv")
all_candles_1h = load_ohlc_from_csv("data/processed/R_10_1H.csv")

# Dynamically filter lookbacks
with open("config/timeframe_windows.yaml") as f:
    tf_cfg = yaml.safe_load(f)["timeframes"]

def filter_lookback(candles, timeframe_key):
    days = tf_cfg[timeframe_key]["lookback_days"]
    cutoff = candles[-1].timestamp - timedelta(days=days)
    return [c for c in candles if c.timestamp >= cutoff]

candles_15m = filter_lookback(all_candles_15m, "15m")
candles_1h  = filter_lookback(all_candles_1h, "1h")

print(f"15m: {len(candles_15m)} candles ({candles_15m[0].timestamp} → {candles_15m[-1].timestamp})")
print(f"1H:  {len(candles_1h)} candles ({candles_1h[0].timestamp} → {candles_1h[-1].timestamp})")

ImportError: cannot import name 'load_data' from 'src.adapters.local_data' (c:\Users\victo\Documents\Projects\chronos-ai\src\adapters\local_data.py)

Cell 2:

In [ ]:
import yaml
from datetime import timedelta
import matplotlib.pyplot as plt
from src.adapters.local_data import load_csv
from src.core.trend_id import identify_trend

# Load original data
all_candles_15m = load_csv("data/processed/R_10_15m.csv")
all_candles_1h = load_csv("data/processed/R_10_1H.csv")

# Dynamically filter lookbacks
with open("config/timeframe_windows.yaml") as f:
    tf_cfg = yaml.safe_load(f)["timeframes"]

def filter_lookback(candles, timeframe_key):
    days = tf_cfg[timeframe_key]["lookback_days"]
    cutoff = candles[-1].timestamp - timedelta(days=days)
    return [c for c in candles if c.timestamp >= cutoff]

candles_15m = filter_lookback(all_candles_15m, "15m")
candles_1h  = filter_lookback(all_candles_1h, "1h")

print(f"15m: {len(candles_15m)} candles ({candles_15m[0].timestamp} → {candles_15m[-1].timestamp})")
print(f"1H:  {len(candles_1h)} candles ({candles_1h[0].timestamp} → {candles_1h[-1].timestamp})")

Cell 3: 

In [ ]:
def draw_trend_chart(candles, result, title):
    # THE FINAL FIX: No more slicing, display all candles exactly matching the index
    WINDOW = len(candles)
    display = candles
    offset = 0
    
    fig_width = max(16, len(display) // 10)
    fig, ax = plt.subplots(figsize=(fig_width, 8))
    
    # Plot closing prices as a simple line for visibility
    prices = [c.close for c in display]
    ax.plot(range(len(display)), prices, color='black', alpha=0.5, linewidth=1)
    
    # Draw legs
    for leg in result["legs"]:
        start_x = leg["start_index"]
        end_x = leg["end_index"] if leg["end_index"] is not None else len(display) - 1
        
        start_y = leg["start_price"]
        end_y = leg["end_price"] if leg["end_price"] is not None else display[-1].close
        
        # Color logic
        if result["trend"] == "down":
            color = "red" if leg["type"] == "impulse" else "green"
        else:
            color = "green" if leg["type"] == "impulse" else "red"
            
        if not leg["confirmed"]:
            ax.plot([start_x, end_x], [start_y, end_y], color='grey', linestyle='--', linewidth=2)
        else:
            ax.plot([start_x, end_x], [start_y, end_y], color=color, linewidth=3)
            # End dot
            ax.scatter(end_x, end_y, color=color, s=100, zorder=5)

    # Format X-axis
    tick_step = max(1, len(display) // 20)
    tick_pos = list(range(0, len(display), tick_step))
    tick_lbl = [display[i].timestamp.strftime("%b %d\n%H:%M") for i in tick_pos]
    
    ax.set_xticks(tick_pos)
    ax.set_xticklabels(tick_lbl, fontsize=7, rotation=45)
    ax.set_title(f"{title} | Trend: {result['trend']} | {len(result['legs'])} legs | Phase: {result['current_phase']}")
    ax.grid(True, alpha=0.3)
    plt.show()

# Run the 15m Chart
draw_trend_chart(candles_15m, result_15m, "R_10 15m")

In [ ]:
# Run the 1H Chart
draw_trend_chart(candles_1h, result_1h, "R_10 1H")